In [ ]:
# This is necessary to recognize the modules
import os
import sys
root_path = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(root_path)
from decimal import Decimal
from theOne import theOne
import pandas as pd
import numpy as np
from core.data_sources.clob import CLOBDataSource
from core.data_structures.candles import Candles

## Load the Candles OHLCV

In [ ]:

CONNECTOR_NAME = "binance"
INTERVALS = "1s"
TRADING_PAIR = "POL-USDT"


In [ ]:
from dataHandler import load_candles_and_orderbook

candles_and_ob_df = load_candles_and_orderbook(CONNECTOR_NAME, INTERVALS, TRADING_PAIR)
candles_and_ob_df.columns

# Backtest:

In [ ]:
def compute_obp(df, idx, n=5, l=5):
    """
    Compute OBP (Order Book Pressure) at a specific index `idx`
    using n candles and l levels deep.
    """
    start = max(0, idx - n + 1)
    bid_sum = 0
    ask_sum = 0
    
    for i in range(start, idx + 1):
        row = df.iloc[i]
        bids = row['bids'][:l]
        asks = row['asks'][:l]
        bid_sum += sum(qty for price, qty in bids)
        ask_sum += sum(qty for price, qty in asks)

    # no orderbook pressure
    if ask_sum ==0 and bid_sum == 0:
        return 0
    
    if ask_sum == 0:
        return 1  # extremely bullish
    
    return bid_sum / ask_sum


In [ ]:
def li_bid_formula(row, obp_sign, tick_size=0.0001, mu=3):
    """Generate bid orders at multiple levels"""
    best_bid = row['bids'][0][0]
    adjustment = obp_sign * mu * tick_size
    p_base = best_bid + adjustment
    return [
        [p_base, 100],
        [p_base - tick_size, 200],
        [p_base - 2 * tick_size, 400]
    ]

def li_ask_formula(row, obp_sign, tick_size=0.0001, mu=3):
    """Generate ask orders at multiple levels"""
    best_ask = row['asks'][0][0]
    adjustment = obp_sign * mu * tick_size
    p_base = best_ask + adjustment
    return [
        [p_base, 100],
        [p_base + tick_size, 200],
        [p_base + 2 * tick_size, 400]
    ]

In [ ]:
# candles_and_ob_df[:50]

## Order fill logic

```pseudo_code
FOR each time step:
    IF it's time to refresh orders:
        Recalculate bid and ask quotes using formula
        Replace outstanding orders with new ones

    Compute market volume changes: 
        Δbuy_volume = next_taker_buy_base_volume - current
        Δsell_volume = next_taker_sell_base_volume - current

    FOR each outstanding bid (sorted descending by price):
        Compute:
            V_above = market bids better than price
            V_same_market = market bids at same price
            own_better = own orders at better price
            volume_before = V_above + V_same_market + own_better

        IF Δsell_volume > volume_before:
            filled = min(my_qty, Δsell_volume - volume_before - V_same_market)
            decrement outstanding_bids[i] by filled

    FOR each outstanding ask (sorted ascending by price):
        Compute:
            V_below = market asks better than price
            V_same_market = market asks at same price
            own_better = own orders at better price
            volume_before = V_below + V_same_market + own_better

        IF Δbuy_volume > volume_before:
            filled = min(my_qty, Δbuy_volume - volume_before - V_same_market)
            decrement outstanding_asks[i] by filled

    Clean up fully-filled orders
    Log fill info with debug metadata

```

In [ ]:
def calculate_fills_Lietal_refresh_debug_fixed(
    candles_and_ob_df, 
    bid_formula=None, 
    ask_formula=None, 
    refresh_interval=5
):
    """
    Calculate order fills for each row in candles_and_ob_df using dynamic pricing formulas.
    FIXED VERSION: Corrects over-filling issues by properly modeling FIFO order execution.
    
    Key fixes:
    1. Proper FIFO queue simulation - orders fill only after preceding orders
    2. Correct remaining volume tracking without double-counting
    3. Separate handling of own orders vs market orders
    4. More realistic fill probability based on order queue position
    """
    debug_records = []
    tick_size = 0.0001
    mu = 2
    n = 5
    l = 5

    outstanding_bids = []
    outstanding_asks = []
    last_refresh = -refresh_interval

    for idx in range(len(candles_and_ob_df) - 1):
        row = candles_and_ob_df.iloc[idx]
        next_row = candles_and_ob_df.iloc[idx + 1]

        # Check volume changes
        buy_vol_changed = row['taker_buy_base_volume'] != next_row['taker_buy_base_volume']
        sell_vol_changed = row['taker_sell_base_volume'] != next_row['taker_sell_base_volume']
        
        # Calculate incremental volumes
        S = next_row['taker_sell_base_volume'] - row['taker_sell_base_volume'] if sell_vol_changed else 0
        B = next_row['taker_buy_base_volume'] - row['taker_buy_base_volume'] if buy_vol_changed else 0

        # Refresh orders every refresh_interval seconds
        order_refreshed = False
        if (idx - last_refresh) >= refresh_interval:
            obp = compute_obp(candles_and_ob_df, idx, n=n, l=l)
            obp_sign = 1 if obp > 1 else (-1 if obp < 1 else 0)
            outstanding_bids = bid_formula(row, obp_sign, tick_size, mu)
            outstanding_asks = ask_formula(row, obp_sign, tick_size, mu)
            outstanding_bids = [[price, qty] for price, qty in outstanding_bids]
            outstanding_asks = [[price, qty] for price, qty in outstanding_asks]
            last_refresh = idx
            order_refreshed = True

        # Process BUY LIMIT ORDERS (filled when market sells occur)
        if outstanding_bids and sell_vol_changed and S > 0:
            # Sort bids by price descending for proper fill priority
            outstanding_bids_sorted = sorted(enumerate(outstanding_bids), 
                                           key=lambda x: x[1][0], reverse=True)
            
            # Track how much market sell volume has been consumed
            consumed_volume = 0
            
            for level, (original_idx, (price, qty)) in enumerate(outstanding_bids_sorted):
                if outstanding_bids[original_idx][1] <= 0:  # Skip already filled orders
                    continue
                
                # Calculate market liquidity that needs to be consumed before this order
                V_above = sum(v for p, v in row['bids'] if p > price)
                V_same_market = sum(v for p, v in row['bids'] if p == price)
                
                # Own better orders that would fill before this one
                own_better = sum(outstanding_bids[j][1] for j in range(len(outstanding_bids)) 
                               if j != original_idx and outstanding_bids[j][0] > price and outstanding_bids[j][1] > 0)
                
                # Total volume that must be consumed before this order gets a chance
                volume_before_this_order = V_above + own_better
                
                # Check if market volume is sufficient to reach this order
                if S <= volume_before_this_order:
                    filled = 0  # Market volume doesn't reach this order
                else:
                    # Volume available for this specific order and same-price market orders
                    available_volume = S - volume_before_this_order
                    
                    # At same price level, assume FIFO with market orders having time priority
                    # Conservative assumption: market orders at same price fill first
                    if available_volume > V_same_market:
                        # Volume left after market orders at same price
                        volume_for_own_orders = available_volume - V_same_market
                        filled = min(outstanding_bids[original_idx][1], volume_for_own_orders)
                    else:
                        # Not enough volume to get through market orders at same price
                        filled = 0
                    
                    # Ensure we don't fill more than available
                    filled = max(0, filled)
                
                # Update the order quantity
                if filled > 0:
                    outstanding_bids[original_idx][1] -= filled
                
                # Record debug info for this order level
                debug_records.append({
                    'timestamp': row['timestamp'],
                    'idx': idx,
                    'side': 'buy',
                    'level': level + 1,
                    'price': price,
                    'qty': qty,
                    'filled': filled,
                    'V_above': V_above,
                    'V_same_market': V_same_market,
                    'own_better': own_better,
                    'volume_before_this_order': volume_before_this_order,
                    'market_sell_volume': S,
                    'volume_changed': sell_vol_changed,
                    'order_refreshed': order_refreshed,
                    'obp': compute_obp(candles_and_ob_df, idx, n=n, l=l) if idx >= n else None,
                    'best_bid': row['bids'][0][0] if row['bids'] else None,
                    'best_ask': row['asks'][0][0] if row['asks'] else None,
                    'spread': (row['asks'][0][0] - row['bids'][0][0]) if (row['bids'] and row['asks']) else None
                })

        # Process SELL LIMIT ORDERS (filled when market buys occur)  
        if outstanding_asks and buy_vol_changed and B > 0:
            # Sort asks by price ascending for proper fill priority
            outstanding_asks_sorted = sorted(enumerate(outstanding_asks), 
                                           key=lambda x: x[1][0])
            
            for level, (original_idx, (price, qty)) in enumerate(outstanding_asks_sorted):
                if outstanding_asks[original_idx][1] <= 0:  # Skip already filled orders
                    continue
                
                # Calculate market liquidity that needs to be consumed before this order
                V_below = sum(v for p, v in row['asks'] if p < price)
                V_same_market = sum(v for p, v in row['asks'] if p == price)
                
                # Own better orders that would fill before this one
                own_better = sum(outstanding_asks[j][1] for j in range(len(outstanding_asks)) 
                               if j != original_idx and outstanding_asks[j][0] < price and outstanding_asks[j][1] > 0)
                
                # Total volume that must be consumed before this order gets a chance
                volume_before_this_order = V_below + own_better
                
                # Check if market volume is sufficient to reach this order
                if B <= volume_before_this_order:
                    filled = 0  # Market volume doesn't reach this order
                else:
                    # Volume available for this specific order and same-price market orders
                    available_volume = B - volume_before_this_order
                    
                    # At same price level, assume FIFO with market orders having time priority
                    if available_volume > V_same_market:
                        # Volume left after market orders at same price
                        volume_for_own_orders = available_volume - V_same_market
                        filled = min(outstanding_asks[original_idx][1], volume_for_own_orders)
                    else:
                        # Not enough volume to get through market orders at same price
                        filled = 0
                    
                    # Ensure we don't fill more than available
                    filled = max(0, filled)
                
                # Update the order quantity
                if filled > 0:
                    outstanding_asks[original_idx][1] -= filled
                
                # Record debug info for this order level
                debug_records.append({
                    'timestamp': row['timestamp'],
                    'idx': idx,
                    'side': 'sell',
                    'level': level + 1,
                    'price': price,
                    'qty': qty,
                    'filled': filled,
                    'V_below': V_below,
                    'V_same_market': V_same_market,
                    'own_better': own_better,
                    'volume_before_this_order': volume_before_this_order,
                    'market_buy_volume': B,
                    'volume_changed': buy_vol_changed,
                    'order_refreshed': order_refreshed,
                    'obp': compute_obp(candles_and_ob_df, idx, n=n, l=l) if idx >= n else None,
                    'best_bid': row['bids'][0][0] if row['bids'] else None,
                    'best_ask': row['asks'][0][0] if row['asks'] else None,
                    'spread': (row['asks'][0][0] - row['bids'][0][0]) if (row['bids'] and row['asks']) else None
                })

        # Clean up filled orders
        outstanding_bids = [order for order in outstanding_bids if order[1] > 0]
        outstanding_asks = [order for order in outstanding_asks if order[1] > 0]

    return debug_records

In [ ]:

# Get debug data with all order information
order_fills_records = calculate_fills_Lietal_refresh_debug_fixed(
    candles_and_ob_df, 
    bid_formula=li_bid_formula, 
    ask_formula=li_ask_formula, 
    refresh_interval=5
)

# Convert to DataFrame for analysis
order_fills_df = pd.DataFrame(order_fills_records)
order_fills_df.columns
order_fills_df

In [ ]:
def add_fill_columns_to_candles(candles_and_ob_df, order_fills_df):
    """
    Add dynamic fill columns to the main candles dataframe.
    
    Parameters:
    -----------
    candles_and_ob_df : pd.DataFrame
        Main dataframe with candle and order book data
    order_fills_df : pd.DataFrame
        DataFrame containing order fill information
        
    Returns:
    --------
    pd.DataFrame
        Enhanced candles dataframe with fill columns added
    """
    
    # Create a copy to avoid modifying the original
    enhanced_df = candles_and_ob_df.copy()
    
    # Determine the maximum number of levels for each side
    max_buy_level = order_fills_df[order_fills_df['side'] == 'buy']['level'].max() if len(order_fills_df[order_fills_df['side'] == 'buy']) > 0 else -1
    max_sell_level = order_fills_df[order_fills_df['side'] == 'sell']['level'].max() if len(order_fills_df[order_fills_df['side'] == 'sell']) > 0 else -1
    
    # Handle case where there are no fills
    if pd.isna(max_buy_level):
        max_buy_level = -1
    if pd.isna(max_sell_level):
        max_sell_level = -1
    
    # Create column names dynamically
    fill_columns = []
    
    # Add buy level columns
    for level in range(1, int(max_buy_level) + 1):
        qty_col = f'buy_L{level}_qty'
        price_col = f'buy_L{level}_price'
        filled_col = f'buy_L{level}_filled'
        fill_columns.extend([qty_col, price_col, filled_col])
    
    # Add sell level columns  
    for level in range(1, int(max_sell_level) + 1):
        qty_col = f'sell_L{level}_qty'
        price_col = f'sell_L{level}_price'
        filled_col = f'sell_L{level}_filled'
        fill_columns.extend([qty_col, price_col, filled_col])
    
    # Initialize all fill columns with NaN
    for col in fill_columns:
        enhanced_df[col] = np.nan
    
    # Process fills and aggregate by timestamp, side, and level
    if len(order_fills_df) > 0:
        # Group fills by timestamp, side, and level
        fill_groups = order_fills_df.groupby(['timestamp', 'side', 'level']).agg({
            'qty': 'sum',  # Sum quantities if multiple fills at same level
            'price': lambda x: np.average(x, weights=order_fills_df.loc[x.index, 'qty']),  # Weighted average price
            'filled': 'sum'  # Sum filled amounts
        }).reset_index()
        
        # Map fills to the enhanced dataframe
        for _, fill_row in fill_groups.iterrows():
            timestamp = fill_row['timestamp']
            side = fill_row['side']
            level = int(fill_row['level'])
            qty = fill_row['qty']
            price = fill_row['price']
            filled = fill_row['filled']
            
            # Find matching timestamp in enhanced_df
            mask = enhanced_df['timestamp'] == timestamp
            
            if mask.any():
                qty_col = f'{side}_L{level}_qty'
                price_col = f'{side}_L{level}_price'
                filled_col = f'{side}_L{level}_filled'
                
                # Set the values
                enhanced_df.loc[mask, qty_col] = qty
                enhanced_df.loc[mask, price_col] = price
                enhanced_df.loc[mask, filled_col] = filled
    
    return enhanced_df

In [ ]:
# Add the fill columns
enhanced_candles_df = add_fill_columns_to_candles(candles_and_ob_df, order_fills_df)
# enhanced_candles_df.to_csv('candles_of_and_fills.csv')
# enhanced_candles_df
# Get summary of what was added
# summary = get_fill_summary(enhanced_candles_df)
# print(f"Added {summary['total_columns_added']} columns for {summary['buy_levels']} buy levels and {summary['sell_levels']} sell levels")

#### Order_fills 
This dataframe contains the order fills for the timestamps where market volume changed and the orders were possibly filled.

## Backtest Data Analysis

In [ ]:
def calculate_portfolio_metrics_fixed(debug_df, initial_base_stock=0, initial_quote_stock=10000, initial_avg_cost=None):
    """
    Calculate and append base stock, quote stock, and PnL columns to debug_df.
    FIXED VERSION - eliminates double counting in PnL calculations.
    
    Parameters:
    -----------
    debug_df : pd.DataFrame
        Debug dataframe from calculate_fills_Lietal_refresh_debug
    initial_base_stock : float
        Starting amount of base asset (default: 0)
    initial_quote_stock : float  
        Starting amount of quote asset (default: 10000)
    initial_avg_cost : float
        Average cost basis for initial base stock (required if initial_base_stock > 0)
    
    Returns:
    --------
    pd.DataFrame
        Original dataframe with added columns:
        - base_stock: Running balance of base asset
        - quote_stock: Running balance of quote asset  
        - trade_pnl: REALIZED PnL from this specific trade (0 for buys, actual PnL for sells)
        - cumulative_realized_pnl: Running cumulative REALIZED PnL only
        - unrealized_pnl: Mark-to-market PnL on base holdings
        - total_pnl: cumulative_realized_pnl + unrealized_pnl
        - total_portfolio_value: Base value + Quote stock at current price
    """
    
    # Create a copy to avoid modifying original
    df = debug_df.copy()
    
    # Sort by timestamp and idx to ensure chronological order
    df = df.sort_values(['timestamp', 'idx', 'side', 'level']).reset_index(drop=True)
    
    # Initialize tracking variables
    base_stock = initial_base_stock
    quote_stock = initial_quote_stock
    cumulative_realized_pnl = 0
    
    # Initialize cost basis tracking
    if initial_base_stock > 0:
        if initial_avg_cost is None:
            raise ValueError("Must provide initial_avg_cost if starting with base_stock > 0")
        avg_cost_basis = initial_avg_cost
        total_cost_basis = initial_base_stock * initial_avg_cost
    else:
        avg_cost_basis = 0.0
        total_cost_basis = 0.0
    
    # Initialize new columns
    df['base_stock'] = 0.0
    df['quote_stock'] = 0.0
    df['trade_pnl'] = 0.0  # This will now only show REALIZED PnL
    df['cumulative_realized_pnl'] = 0.0
    df['unrealized_pnl'] = 0.0
    df['total_pnl'] = 0.0
    df['total_portfolio_value'] = 0.0
    df['avg_cost_basis'] = 0.0
    
    for i in range(len(df)):
        row = df.iloc[i]
        
        # Calculate trade PnL and update positions if there was a fill
        trade_pnl = 0  # Default: no realized PnL
        
        if row['filled'] > 0:
            fill_amount = row['filled']
            fill_price = row['price']
            
            if row['side'] == 'buy':
                # BUY ORDER FILLED
                # This is NOT a loss - it's converting cash to inventory
                quote_spent = fill_amount * fill_price
                quote_stock -= quote_spent
                base_stock += fill_amount
                
                # Update cost basis using weighted average
                total_cost_basis += quote_spent
                avg_cost_basis = total_cost_basis / base_stock if base_stock > 0 else 0
                
                # CRITICAL FIX: No realized PnL on buy orders
                trade_pnl = 0  # Buying is not a profit or loss event
                
            elif row['side'] == 'sell':
                # SELL ORDER FILLED
                # This IS a realized PnL event
                quote_gained = fill_amount * fill_price
                quote_stock += quote_gained
                base_stock -= fill_amount
                
                # Calculate realized PnL based on cost basis
                if avg_cost_basis > 0:
                    cost_of_sold = fill_amount * avg_cost_basis
                    trade_pnl = quote_gained - cost_of_sold  # This is ACTUAL realized PnL
                    
                    # Update total cost basis
                    total_cost_basis -= cost_of_sold
                    # avg_cost_basis stays the same (FIFO assumption)
                else:
                    # If no cost basis, treat entire proceeds as profit
                    # (This handles edge cases like short selling or starting with base assets at zero cost)
                    trade_pnl = quote_gained
        
        # Update cumulative REALIZED PnL only
        cumulative_realized_pnl += trade_pnl
        
        # Calculate unrealized PnL on remaining base holdings
        current_price = row['best_bid'] if row['best_bid'] is not None else row['price']
        
        if base_stock > 0 and avg_cost_basis > 0:
            # Unrealized PnL = (current_price - avg_cost_basis) * quantity_held
            unrealized_pnl = base_stock * (current_price - avg_cost_basis)
        else:
            unrealized_pnl = 0
        
        # Total PnL = Realized PnL + Unrealized PnL
        total_pnl = cumulative_realized_pnl + unrealized_pnl
        
        # Total portfolio value
        base_value = base_stock * current_price
        total_portfolio_value = base_value + quote_stock
        
        # Update dataframe
        df.at[i, 'base_stock'] = base_stock
        df.at[i, 'quote_stock'] = quote_stock
        df.at[i, 'trade_pnl'] = trade_pnl
        df.at[i, 'cumulative_realized_pnl'] = cumulative_realized_pnl
        df.at[i, 'unrealized_pnl'] = unrealized_pnl
        df.at[i, 'total_pnl'] = total_pnl
        df.at[i, 'total_portfolio_value'] = total_portfolio_value
        df.at[i, 'avg_cost_basis'] = avg_cost_basis
    
    return df


def portfolio_summary_fixed(df_with_portfolio):
    """
    Generate a summary of portfolio performance using the FIXED calculation method
    """
    if df_with_portfolio.empty:
        return "No data to analyze"
    
    # Get final values
    final_row = df_with_portfolio.iloc[-1]
    initial_row = df_with_portfolio.iloc[0]
    
    # Get filled trades only
    trades = df_with_portfolio[df_with_portfolio['filled'] > 0]
    buy_trades = trades[trades['side'] == 'buy']
    sell_trades = trades[trades['side'] == 'sell']
    
    # Calculate initial portfolio value (for return calculation)
    initial_portfolio_value = initial_row['total_portfolio_value']
    
    summary = {
        'total_trades': len(trades),
        'buy_trades': len(buy_trades),
        'sell_trades': len(sell_trades),
        'total_base_bought': buy_trades['filled'].sum(),
        'total_base_sold': sell_trades['filled'].sum(),
        'total_quote_spent': (buy_trades['filled'] * buy_trades['price']).sum(),
        'total_quote_received': (sell_trades['filled'] * sell_trades['price']).sum(),
        
        # Final positions
        'final_base_stock': final_row['base_stock'],
        'final_quote_stock': final_row['quote_stock'],
        'final_avg_cost_basis': final_row['avg_cost_basis'],
        
        # PnL breakdown (the key improvement)
        'cumulative_realized_pnl': final_row['cumulative_realized_pnl'],
        'unrealized_pnl': final_row['unrealized_pnl'],
        'total_pnl': final_row['total_pnl'],
        
        # Portfolio metrics
        'initial_portfolio_value': initial_portfolio_value,
        'final_portfolio_value': final_row['total_portfolio_value'],
        'total_return_pct': ((final_row['total_portfolio_value'] - initial_portfolio_value) / initial_portfolio_value * 100) if initial_portfolio_value > 0 else 0,
        
        # Trade analysis (only sells generate realized PnL)
        'profitable_sells': len(sell_trades[sell_trades['trade_pnl'] > 0]),
        'losing_sells': len(sell_trades[sell_trades['trade_pnl'] < 0]),
        'avg_realized_pnl_per_sell': sell_trades['trade_pnl'].mean() if len(sell_trades) > 0 else 0,
        'max_sell_profit': sell_trades['trade_pnl'].max() if len(sell_trades) > 0 else 0,
        'max_sell_loss': sell_trades['trade_pnl'].min() if len(sell_trades) > 0 else 0,
        
        # Net position
        'net_base_traded': buy_trades['filled'].sum() - sell_trades['filled'].sum(),
    }
    
    return summary


# Example usage and validation
def validate_pnl_calculation(df_with_portfolio):
    """
    Validate that PnL calculations make sense
    """
    final_row = df_with_portfolio.iloc[-1]
    initial_row = df_with_portfolio.iloc[0]
    
    # Manual calculation
    trades = df_with_portfolio[df_with_portfolio['filled'] > 0]
    buy_trades = trades[trades['side'] == 'buy']
    sell_trades = trades[trades['side'] == 'sell']
    
    total_spent = (buy_trades['filled'] * buy_trades['price']).sum()
    total_received = (sell_trades['filled'] * sell_trades['price']).sum()
    
    print("=== PnL Validation ===")
    print(f"Total quote spent on buys: ${total_spent:.2f}")
    print(f"Total quote received from sells: ${total_received:.2f}")
    print(f"Net cash flow: ${total_received - total_spent:.2f}")
    print(f"Realized PnL (from our calculation): ${final_row['cumulative_realized_pnl']:.2f}")
    print(f"Unrealized PnL: ${final_row['unrealized_pnl']:.2f}")
    print(f"Total PnL: ${final_row['total_pnl']:.2f}")
    
    # The key insight: realized PnL should match the profit/loss from sells only
    manual_realized_pnl = sell_trades['trade_pnl'].sum()
    print(f"Manual realized PnL calculation: ${manual_realized_pnl:.2f}")
    
    if abs(manual_realized_pnl - final_row['cumulative_realized_pnl']) < 0.01:
        print("✅ Realized PnL calculation is correct!")
    else:
        print("❌ Realized PnL calculation has errors!")

In [ ]:
order_fills_df_with_portfolio = calculate_portfolio_metrics_fixed(
    order_fills_df, 
    initial_base_stock=5000,
    initial_quote_stock=20000,
    initial_avg_cost=0.2520  # You need to specify the cost basis for initial inventory
)
order_fills_df_with_portfolio

validate_pnl_calculation(order_fills_df_with_portfolio)

As things stand now my backtest supports short positions implicitly thus that needs to be removed. 